Create "Passive Voice" Entries for an Instruction Dataset

In [ ]:
from importlib.metadata import version

pkgs = ["openai",  # OpenAI API（调用 GPT 模型）
        "tqdm",    # Progress bar（进度条）
       ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
import json
from openai import OpenAI

# 从本地 config.json 读取 OpenAI API key（把 "sk-..." 换成你自己的真实 key）
# key 从 https://platform.openai.com/api-keys 获取；用文件而非硬编码，避免把密钥写进 notebook
with open("config.json", "r") as config_file:
    config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]

client = OpenAI(api_key=api_key)  # 创建 OpenAI 客户端

In [ ]:
# 封装一次 ChatGPT 调用：temperature=0 让输出尽量确定、可复现
def run_chatgpt(prompt, client, model="gpt-4-turbo"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return response.choices[0].message.content


# Prepare input（先用一句话测试主动->被动语态转换）
sentence = "I ate breakfast"
prompt = f"Convert the following sentence to passive voice: '{sentence}'"
run_chatgpt(prompt, client)

In [ ]:
import json

json_file = "instruction-examples.json"  # 待处理的指令样本文件

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))  # 打印条目数

In [ ]:
# 先拿前 5 条试跑：把每条的 output 用 GPT 改写成被动语态，直观检查效果
for entry in json_data[:5]:
    text = entry["output"]
    prompt = f"Without adding any response or explanation, convert the following text to passive voice: {text}"

    print("\nInput:")
    print(">>", text)
    print("\nOutput:")
    print(">>", run_chatgpt(prompt, client))
    print("\n-------------------------")

In [ ]:
from tqdm import tqdm  # a progress bar tool（进度条）


# 前 5 条：把被动语态改写结果写入新字段 output_2（用于数据增强/对照）
for i, entry in tqdm(enumerate(json_data[:5]), total=len(json_data[:5])):
    text = entry["output"]
    prompt = f"Without adding any response or explanation, convert the following text to passive voice: {text}"
    json_data[i]["output_2"] = run_chatgpt(prompt, client)

In [ ]:
# 对「全部」条目执行同样的被动语态改写，结果存入各自的 output_2 字段
for i, entry in tqdm(enumerate(json_data), total=len(json_data)):
    text = entry["output"]
    prompt = f"Without adding any response or explanation, convert the following text to passive voice: {text}"
    json_data[i]["output_2"] = run_chatgpt(prompt, client)

In [ ]:
# 把带有 output_2(被动语态版) 的数据另存为 *-modified.json
new_json_file = json_file.replace(".json", "-modified.json")


with open(new_json_file, "w") as file:
    json.dump(json_data, file, indent=4)  # "indent" for pretty-printing（缩进美化）